In [11]:
import pandas as pd

df = pd.read_csv("ENSO_data.csv", encoding="latin1")

print(df.head())

   Unnamed: 0      Date  Year Month  Global Temperature Anomaly  Nino 1+2 SST  \
0           0  1/1/1950  1950   JAN                       -0.26         23.01   
1           1  2/1/1950  1950   FEB                       -0.27         24.32   
2           2  3/1/1950  1950   MAR                       -0.07         25.11   
3           3  4/1/1950  1950   APR                       -0.21         23.63   
4           4  5/1/1950  1950   MAY                       -0.11         22.68   

   Nino 1+2 SST Anomaly  Nino 3 SST  Nino 3 SST Anomaly  Nino 3.4 SST  ...  \
0                 -1.55       23.56               -2.10         24.55  ...   
1                 -1.78       24.89               -1.52         25.06  ...   
2                 -1.38       26.36               -0.84         25.87  ...   
3                 -1.90       26.44               -1.14         26.28  ...   
4                 -1.74       25.69               -1.57         26.18  ...   

   BEST  Season (2-Month)  MEI.ext  MEI.v2  

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 924 entries, 0 to 923
Data columns (total 29 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Unnamed: 0                         924 non-null    int64  
 1   Date                               924 non-null    object 
 2   Year                               924 non-null    int64  
 3   Month                              924 non-null    object 
 4   Global Temperature Anomaly         917 non-null    float64
 5   Nino 1+2 SST                       917 non-null    float64
 6   Nino 1+2 SST Anomaly               917 non-null    float64
 7   Nino 3 SST                         917 non-null    float64
 8   Nino 3 SST Anomaly                 917 non-null    float64
 9   Nino 3.4 SST                       917 non-null    float64
 10  Nino 3.4 SST Anomaly               917 non-null    float64
 11  Nino 4 SST                         917 non-null    float64

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


model = df.copy()

print("Original shape:", model.shape)
print("Working copy shape:", model.shape)

Original shape: (924, 28)
Working copy shape: (924, 28)


In [37]:

model = model.drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

print("Shape after removing index column:", model.shape)

Shape after removing index column: (924, 28)


In [38]:

print("Duplicate rows:", model.duplicated().sum())


model = model.drop_duplicates()

print("Shape after removing duplicates:", model.shape)

Duplicate rows: 0
Shape after removing duplicates: (924, 28)


In [39]:

model["Date"] = pd.to_datetime(
    model["Date"],
    
)

print("Invalid or missing dates:", model["Date"].isna().sum())
print("Repeated dates:", model["Date"].duplicated().sum())

Invalid or missing dates: 0
Repeated dates: 0


In [46]:

print(model.dtypes)

Date                                 datetime64[ns]
Year                                          int64
Month                                        object
Global Temperature Anomaly                  float64
Nino 1+2 SST                                float64
Nino 1+2 SST Anomaly                        float64
Nino 3 SST                                  float64
Nino 3 SST Anomaly                          float64
Nino 3.4 SST                                float64
Nino 3.4 SST Anomaly                        float64
Nino 4 SST                                  float64
Nino 4 SST Anomaly                          float64
   TNI                                      float64
SOI                                         float64
  OLR                                       float64
   PNA                                      float64
 NOI                                        float64
ENSO Precipitation Index                    float64
BEST                                        float64
Season (2-Mo

In [52]:
print("Original Month values:")
print(df["Month"].value_counts(dropna=False).head(20))

print("\nData type in original DataFrame:")
print(df["Month"].dtype)

print("\nFirst 15 values:")
print(df["Month"].head(15).tolist())

Original Month values:
Month
JAN    77
FEB    77
MAR    77
APR    77
MAY    77
JUN    77
JUL    77
AUG    77
SEP    77
OCT    77
NOV    77
DEC    77
Name: count, dtype: int64

Data type in original DataFrame:
object

First 15 values:
['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC', 'JAN', 'FEB', 'MAR']


In [53]:
month_map = {
    "JAN": 1,
    "FEB": 2,
    "MAR": 3,
    "APR": 4,
    "MAY": 5,
    "JUN": 6,
    "JUL": 7,
    "AUG": 8,
    "SEP": 9,
    "OCT": 10,
    "NOV": 11,
    "DEC": 12
}
model = df.copy()


model = model.drop(columns=["Unnamed: 0"], errors="ignore")


model["Month"] = (
    model["Month"].str.strip().str.upper().map(month_map)
)
print("Unique month values:")
print(sorted(model["Month"].dropna().unique()))

print("\nMonth data type:")
print(model["Month"].dtype)

print("\nMissing month values:")
print(model["Month"].isna().sum())

print("\nFirst 15 rows:")
print(model[["Date", "Month"]].head(15))

Unique month values:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Month data type:
int64

Missing month values:
0

First 15 rows:
         Date  Month
0    1/1/1950      1
1    2/1/1950      2
2    3/1/1950      3
3    4/1/1950      4
4    5/1/1950      5
5    6/1/1950      6
6    7/1/1950      7
7    8/1/1950      8
8    9/1/1950      9
9   10/1/1950     10
10  11/1/1950     11
11  12/1/1950     12
12   1/1/1951      1
13   2/1/1951      2
14   3/1/1951      3


In [55]:

phase_columns = [
    col for col in df.columns
    if "ENSO Phase" in col and "ONI-based" in col
]

print("Matching columns:", phase_columns)

target = phase_columns[0]

print("Original ENSO phase labels:")
print(df[target].value_counts(dropna=False))

print("\nUnique labels:")
print(df[target].unique())

print("\nData type:")
print(df[target].dtype)

Matching columns: ['ENSO Phase\x96Intensity (ONI-based)', 'ENSO Phase\x96Intensity (RONI-based)']
Original ENSO phase labels:
ENSO PhaseIntensity (ONI-based)
N      276
WE     144
WL     144
ME      84
SL      84
ML      72
SE      72
VSE     36
NaN     12
Name: count, dtype: int64

Unique labels:
['N' 'ME' 'WE' 'WL' 'ML' 'SE' 'SL' 'VSE' nan]

Data type:
object


In [56]:
phase_map = {
    "N": 0,
    "WE": 1,
    "ME": 1,
    "SE": 1,
    "VSE": 1,
    "WL": -1,
    "ML": -1,
    "SL": -1
}

model["ENSO_Target"] = (
    model[target].str.strip().map(phase_map)
)

print("Encoded class counts:")
print(model["ENSO_Target"].value_counts(dropna=False))

print("\nUnique encoded values:")
print(model["ENSO_Target"].unique())

print("\nMissing target values:")
print(model["ENSO_Target"].isna().sum())

Encoded class counts:
ENSO_Target
 1.0    336
-1.0    300
 0.0    276
 NaN     12
Name: count, dtype: int64

Unique encoded values:
[ 0.  1. -1. nan]

Missing target values:
12


In [57]:
model = model.dropna(subset=["ENSO_Target"]).copy()

print("Shape after removing missing targets:", model.shape)
print("Missing targets:", model["ENSO_Target"].isna().sum())

Shape after removing missing targets: (912, 29)
Missing targets: 0


In [59]:
print(model.isnull().sum().sort_values(ascending=False))
model["Date"] = pd.to_datetime(model["Date"], errors="coerce")

model = model.sort_values("Date").reset_index(drop=True)

ENSO Precipitation Index             374
MEI.v2                               348
  OLR                                309
MEI.ext                              240
SOI                                   12
 NOI                                   2
   TNI                                 2
ENSO PhaseIntensity (RONI-based)      0
ENSO PhaseIntensity (ONI-based)       0
Season (12-Month)                      0
RONI                                   0
  ONI                                  0
Season (3-Month)                       0
Season (2-Month)                       0
BEST                                   0
Date                                   0
   PNA                                 0
Year                                   0
Nino 4 SST Anomaly                     0
Nino 4 SST                             0
Nino 3.4 SST Anomaly                   0
Nino 3.4 SST                           0
Nino 3 SST Anomaly                     0
Nino 3 SST                             0
Nino 1+2 SST Ano

In [60]:
import pandas as pd


model["Date"] = pd.to_datetime(model["Date"])
model = model.sort_values("Date").reset_index(drop=True)


features = [
    "Nino 1+2 SST Anomaly",
    "Nino 3 SST Anomaly",
    "Nino 3.4 SST Anomaly",
    "Nino 4 SST Anomaly",
    "SOI"
]

model["Future_ENSO_Target"] = model["ENSO_Target"].shift(-1)


X = model[features]
y = model["Future_ENSO_Target"]


X = X.iloc[:-1].copy()
y = y.iloc[:-1].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nSelected features:")
print(X.head())

print("\nTarget values:")
print(y.head())

Features shape: (911, 5)
Target shape: (911,)

Selected features:
   Nino 1+2 SST Anomaly  Nino 3 SST Anomaly  Nino 3.4 SST Anomaly  \
0                 -1.55               -2.10                 -1.99   
1                 -1.78               -1.52                 -1.69   
2                 -1.38               -0.84                 -1.42   
3                 -1.90               -1.14                 -1.54   
4                 -1.74               -1.57                 -1.75   

   Nino 4 SST Anomaly  SOI  
0               -1.38  NaN  
1               -1.53  NaN  
2               -1.80  NaN  
3               -1.73  NaN  
4               -1.18  NaN  

Target values:
0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Future_ENSO_Target, dtype: float64


In [64]:

months = model["Date"].dt.year * 12 + model["Date"].dt.month

month_diff = months.diff().dropna()

gaps = month_diff[month_diff != 1]

print("Number of missing-month intervals:", len(gaps))

if len(gaps) == 0:
    print("All dates are consecutive months.")
else:
    print("Intervals that need checking:")
    print(gaps.head(10))


    bad_indices = gaps.index[:10]
    print(model.loc[bad_indices - 1, ["Date"]].rename(
        columns={"Date": "Previous date"}
    ).reset_index(drop=True))
    print(model.loc[bad_indices, ["Date"]].rename(
        columns={"Date": "Current date"}
    ).reset_index(drop=True))

Number of missing-month intervals: 0
All dates are consecutive months.


In [65]:

split_index = int(len(X) * 0.8)


X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()


y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)


print("\nTraining period:")
print(model.loc[X_train.index[0], "Date"], "to",
      model.loc[X_train.index[-1], "Date"])

print("\nTesting period:")
print(model.loc[X_test.index[0], "Date"], "to",
      model.loc[X_test.index[-1], "Date"])

Training features: (728, 5)
Testing features: (183, 5)
Training targets: (728,)
Testing targets: (183,)

Training period:
1950-01-01 00:00:00 to 2010-08-01 00:00:00

Testing period:
2010-09-01 00:00:00 to 2025-11-01 00:00:00


In [68]:
import pandas as pd

missing_before = X_train.isnull().sum()


missing_after = pd.DataFrame(
    X_train_imputed,
    columns=X.columns
).isnull().sum()

missing_comparison = pd.DataFrame({
    "Before Imputation": missing_before,
    "After Imputation": missing_after
})

print(missing_comparison)

                      Before Imputation  After Imputation
Nino 1+2 SST Anomaly                  0                 0
Nino 3 SST Anomaly                    0                 0
Nino 3.4 SST Anomaly                  0                 0
Nino 4 SST Anomaly                    0                 0
SOI                                  12                 0


In [69]:
from sklearn.preprocessing import StandardScaler
import pandas as pd


scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(X_train_imputed)


X_test_scaled = scaler.transform(X_test_imputed)


print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled testing data shape:", X_test_scaled.shape)


print("\nFirst five rows of scaled training data:")
print(pd.DataFrame(
    X_train_scaled,
    columns=X.columns
).head())

Scaled training data shape: (728, 5)
Scaled testing data shape: (183, 5)

First five rows of scaled training data:
   Nino 1+2 SST Anomaly  Nino 3 SST Anomaly  Nino 3.4 SST Anomaly  \
0             -1.127302           -2.061008             -2.058417   
1             -1.336273           -1.413913             -1.712070   
2             -0.972845           -0.655249             -1.400358   
3             -1.445302           -0.989953             -1.538897   
4             -1.299931           -1.469697             -1.781339   

   Nino 4 SST Anomaly       SOI  
0           -1.595090  0.013612  
1           -1.819803  0.013612  
2           -2.224286  0.013612  
3           -2.119420  0.013612  
4           -1.295473  0.013612  


In [70]:
import numpy as np

class PegasosSVM:
    def __init__(self, lambda_param=0.01, T=1000):
        self.lambda_param = lambda_param
        self.T = T
        self.w = None
        self.b = 0.0

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        n_samples, n_features = X.shape

       
        self.w = np.zeros(n_features)
        self.b = 0.0

        rng = np.random.default_rng(42)

      
        for t in range(1, self.T + 1):

           
            i = rng.integers(0, n_samples)
            x_i = X[i]
            y_i = y[i]

          
            eta = 1 / (self.lambda_param * t)

        
            margin = y_i * (np.dot(self.w, x_i) + self.b)

            self.w = (1 - eta * self.lambda_param) * self.w

           
            if margin < 1:
                self.w += eta * y_i * x_i
                self.b += eta * y_i

            
            norm_w = np.linalg.norm(self.w)
            limit = 1 / np.sqrt(self.lambda_param)

            if norm_w > limit:
                self.w = self.w * (limit / norm_w)

        return self

    def decision_function(self, X):
        X = np.asarray(X, dtype=float)
        return np.dot(X, self.w) + self.b

    def predict(self, X):
        scores = self.decision_function(X)
        return np.where(scores >= 0, 1, -1)

In [71]:
class PegasosMultiClass:
    def __init__(self, lambda_param=0.01, T=1000):
        self.lambda_param = lambda_param
        self.T = T
        self.classes = np.array([-1, 0, 1])
        self.models = {}

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)

        for cls in self.classes:
          
            binary_y = np.where(y == cls, 1, -1)

            model = PegasosSVM(
                lambda_param=self.lambda_param,
                T=self.T
            )
            model.fit(X, binary_y)
            self.models[cls] = model

        return self

    def decision_function(self, X):
        return np.column_stack([
            self.models[cls].decision_function(X)
            for cls in self.classes
        ])

    def predict(self, X):
        scores = self.decision_function(X)
        return self.classes[np.argmax(scores, axis=1)]

In [72]:

model_svm = PegasosMultiClass(
    lambda_param=0.01,
    T=1000
)


model_svm.fit(X_train_scaled, y_train.to_numpy())

print("Pegasos SVM training complete!")

Pegasos SVM training complete!


In [75]:

y_pred = model_svm.predict(X_test_scaled)

print("Predictions completed successfully!")
print("First 10 predictions:")
print(y_pred[:10])

Predictions completed successfully!
First 10 predictions:
[ 1  1  1  1  1  1  1  1 -1 -1]


In [76]:
import pandas as pd

comparison = pd.DataFrame({
    "Actual Phase": y_test.to_numpy(),
    "Predicted Phase": y_pred
})


phase_names = {
    -1: "La Niña",
     0: "Neutral",
     1: "El Niño"
}

comparison["Actual Phase"] = comparison["Actual Phase"].map(phase_names)
comparison["Predicted Phase"] = comparison["Predicted Phase"].map(phase_names)

print(comparison.head(15))

   Actual Phase Predicted Phase
0       La Niña         El Niño
1       La Niña         El Niño
2       La Niña         El Niño
3       La Niña         El Niño
4       La Niña         El Niño
5       La Niña         El Niño
6       La Niña         El Niño
7       La Niña         El Niño
8       La Niña         La Niña
9       La Niña         La Niña
10      La Niña         El Niño
11      La Niña         La Niña
12      La Niña         El Niño
13      La Niña         El Niño
14      La Niña         El Niño


In [77]:
print("Predicted ENSO phase counts:")
print(comparison["Predicted Phase"].value_counts())

Predicted ENSO phase counts:
Predicted Phase
El Niño    115
La Niña     47
Neutral     21
Name: count, dtype: int64


In [78]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Pegasos SVM Test Accuracy:",
      round(accuracy * 100, 2), "%")

Pegasos SVM Test Accuracy: 25.68 %


In [103]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

lambda_values = [0.0001, 0.001, 0.01, 0.1, 1.0]
T_values = [1000, 3000, 5000, 10000]

results = []

for lam in lambda_values:
    for T in T_values:

        model = PegasosMultiClass(
            lambda_param=lam,
            T=T
        )

        model.fit(
            X_train_lag.to_numpy(),
            y_train_lag.to_numpy()
        )

        predictions = model.predict(
            X_test_lag.to_numpy()
        )

        accuracy = accuracy_score(
            y_test_lag,
            predictions
        )

        macro_f1 = f1_score(
            y_test_lag,
            predictions,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        )

        results.append({
            "lambda": lam,
            "T": T,
            "Accuracy": accuracy,
            "Macro F1": macro_f1
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Macro F1",
    ascending=False
)

print(results_df.to_string(index=False))

 lambda     T  Accuracy  Macro F1
 0.0001 10000  0.558011  0.540602
 0.0010  5000  0.546961  0.497571
 0.0100  5000  0.546961  0.452899
 0.0001  5000  0.475138  0.450038
 0.0010 10000  0.546961  0.422585
 0.0100  3000  0.535912  0.420846
 0.0100 10000  0.546961  0.419321
 0.0001  3000  0.486188  0.413990
 0.1000  5000  0.524862  0.413383
 0.1000 10000  0.535912  0.411180
 0.0010  1000  0.447514  0.408882
 0.1000  1000  0.491713  0.402381
 0.1000  3000  0.508287  0.400266
 0.0001  1000  0.430939  0.397631
 0.0010  3000  0.486188  0.394315
 1.0000  3000  0.486188  0.384298
 1.0000 10000  0.491713  0.378510
 1.0000  1000  0.464088  0.374691
 1.0000  5000  0.480663  0.369093
 0.0100  1000  0.403315  0.347233


In [104]:
from sklearn.metrics import classification_report, confusion_matrix

best_pegasos = PegasosMultiClass(
    lambda_param=0.0001,
    T=10000
)

best_pegasos.fit(
    X_train_lag.to_numpy(),
    y_train_lag.to_numpy()
)

best_pred = best_pegasos.predict(
    X_test_lag.to_numpy()
)

print(classification_report(
    y_test_lag,
    best_pred,
    labels=[-1, 0, 1],
    target_names=["La Niña", "Neutral", "El Niño"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(
    y_test_lag,
    best_pred,
    labels=[-1, 0, 1],
))

              precision    recall  f1-score   support

     La Niña       0.69      0.42      0.53        73
     Neutral       0.37      0.42      0.39        48
     El Niño       0.61      0.83      0.70        60

    accuracy                           0.56       181
   macro avg       0.56      0.56      0.54       181
weighted avg       0.58      0.56      0.55       181

Confusion matrix:
[[31 25 17]
 [13 20 15]
 [ 1  9 50]]


In [105]:
from sklearn.metrics import (
    accuracy_score,
    mean_squared_error,
    mean_absolute_error,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np


accuracy = accuracy_score(y_test_lag, best_pred)

error_rate = 1 - accuracy


mse = mean_squared_error(y_test_lag, best_pred)


rmse = np.sqrt(mse)


mae = mean_absolute_error(y_test_lag, best_pred)


macro_f1 = f1_score(
    y_test_lag,
    best_pred,
    labels=[-1, 0, 1],
    average="macro",
    zero_division=0
)

# Display results
print("\n===== PEGASOS SVM EVALUATION =====")
print(f"Accuracy:   {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Error Rate: {error_rate:.4f} ({error_rate * 100:.2f}%)")
print(f"MSE:        {mse:.4f}")
print(f"RMSE:       {rmse:.4f}")
print(f"MAE:        {mae:.4f}")
print(f"Macro F1:   {macro_f1:.4f}")


===== PEGASOS SVM EVALUATION =====
Accuracy:   0.5580 (55.80%)
Error Rate: 0.4420 (44.20%)
MSE:        0.7403
RMSE:       0.8604
MAE:        0.5414
Macro F1:   0.5406
